In [1]:
import gc
from datetime import date, datetime, timedelta
from pathlib import Path
 
import polars as pl
import requests
 
GTFS = Path("raw/processed_gtfs")
RT = Path("raw")
OUT = Path("raw/processed_gtfs/daily")
OUT.mkdir(parents=True, exist_ok=True)
 
DATE_START = date(2026, 3, 3)
DATE_END = date(2026, 7, 3)
 
TRIP_ID_PATTERN = r"^[A-Z]{2}_([A-Z0-9]+)-(\w+?)-(\d+)_([A-Z0-9+]+)_(\d+)$"
 
FEATURES = [
    "route_id", "direction_id", "shape_id", "service_id",
    "stop_sequence", "trip_progress",
    "hour", "weekday", "month", "is_peak",
    "scheduled_arrival", "scheduled_departure", "scheduled_segment_time",
    "stop_lat", "stop_lon",
    "latitude", "longitude", "bearing",
    "temperature_c", "precipitation_mm", "snowfall_cm", "windspeed_kmh",
    "is_raining", "is_snowing", "is_fog", "weathercode",
    "segment_length", "scheduled_segment_speed_mps",
    "upstream_delay_seconds", "speed_mps", "headway_seconds",
    "is_weekend", "is_federal_holiday", "is_school_day", "has_major_event",
    "ridership", "transfers",
]
TARGET = "travel_time"
ID_COLS = ["trip_id", "start_date"]
 

In [2]:
def gtfs_to_seconds(col: str) -> pl.Expr:
    p = pl.col(col).str.split(":")
    return p.list.get(0).cast(pl.Int32) * 3600 + p.list.get(1).cast(pl.Int32) * 60 + p.list.get(2).cast(pl.Int32)
 
 
def parse_trip_id(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 2).alias("_service_day"),
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 3).alias("_origin_secs"),
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 5).alias("_trip_num"),
    ])
 
 
def haversine_expr(lat1: pl.Expr, lon1: pl.Expr, lat2: pl.Expr, lon2: pl.Expr) -> pl.Expr:
    R = 6371000.0
    lat1r, lat2r = lat1.radians(), lat2.radians()
    dlat = (lat2 - lat1).radians()
    dlon = (lon2 - lon1).radians()
    a = (dlat / 2).sin() ** 2 + lat1r.cos() * lat2r.cos() * (dlon / 2).sin() ** 2
    return 2 * R * a.sqrt().arcsin()
 
 
def resolve_service_dates(calendar_df: pl.DataFrame, calendar_dates_df: pl.DataFrame) -> pl.DataFrame:
    weekday_cols = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
    rows = []
    for row in calendar_df.iter_rows(named=True):
        start = date(int(str(row["start_date"])[:4]), int(str(row["start_date"])[4:6]), int(str(row["start_date"])[6:8]))
        end = date(int(str(row["end_date"])[:4]), int(str(row["end_date"])[4:6]), int(str(row["end_date"])[6:8]))
        d = start
        while d <= end:
            if row[weekday_cols[d.weekday()]] == 1:
                rows.append({"service_id": row["service_id"], "date": int(d.strftime("%Y%m%d"))})
            d += timedelta(days=1)
    base = pl.DataFrame(rows)
    additions = calendar_dates_df.filter(pl.col("exception_type") == 1).select(["service_id", "date"])
    removals = calendar_dates_df.filter(pl.col("exception_type") == 2).select(["service_id", "date"])
    resolved = pl.concat([base, additions]).unique()
    return resolved.join(removals, on=["service_id", "date"], how="anti")
 

In [3]:
# =====================================================================
# 1. Load small reference tables ONCE — reused across every day, never
#    reloaded or duplicated inside the loop.
# =====================================================================
print("Loading static reference tables...")
 
routes = pl.read_parquet(GTFS / "routes.parquet")
trips = pl.read_parquet(GTFS / "trips.parquet")
stops = pl.read_parquet(GTFS / "stops.parquet").with_columns(pl.col("stop_id").cast(pl.Utf8))
stop_times = pl.read_parquet(GTFS / "stop_times.parquet").with_columns(
    pl.col("stop_sequence").cast(pl.UInt32),
    pl.col("stop_id").cast(pl.Utf8),
    gtfs_to_seconds("arrival_time").alias("scheduled_arrival"),
    gtfs_to_seconds("departure_time").alias("scheduled_departure"),
)
 
calendar = pl.read_parquet(GTFS / "calendar.parquet")
calendar_dates = pl.read_parquet(GTFS / "calendar_dates.parquet")
service_dates = resolve_service_dates(calendar, calendar_dates)
 
segment_network = pl.read_parquet("processed/segment_network.parquet").with_columns([
    pl.col("stop_id").cast(pl.Utf8),
    pl.col("next_stop_id").cast(pl.Utf8),
])
 
calendar_features = (
    pl.read_parquet("calendar_dataset.parquet")
    .with_columns(pl.col("service_date").str.strptime(pl.Date, "%Y-%m-%d"))
    .select(["service_date", "is_weekend", "is_federal_holiday", "is_school_day", "major_event_count"])
    .with_columns((pl.col("major_event_count") > 0).alias("has_major_event"))
)
 
ridership_raw = pl.read_csv(
    "MTA_Bus_Hourly_Ridership_Mar_July.csv",
    schema_overrides={"ridership": pl.Utf8, "transfers": pl.Utf8},
).with_columns(
    pl.col("transit_timestamp").str.strptime(pl.Datetime, "%m/%d/%Y %I:%M:%S %p").alias("_ts"),
    pl.col("ridership").str.replace_all(",", "").cast(pl.Int64),
    pl.col("transfers").str.replace_all(",", "").cast(pl.Int64),
).with_columns([
    pl.col("_ts").dt.date().alias("service_date"),
    pl.col("_ts").dt.hour().alias("hour"),
])
ridership_hourly = ridership_raw.group_by(["bus_route", "service_date", "hour"]).agg([
    pl.col("ridership").sum(),
    pl.col("transfers").sum(),
])
 
trips_parsed = parse_trip_id(trips)
 
# Weather is one API call for the whole range (hourly rows only — tiny),
# not per day.
STATION_LAT, STATION_LON = 40.7829, -73.9654
start_fmt, end_fmt = DATE_START.isoformat(), DATE_END.isoformat()
print(f"Fetching weather for {start_fmt} to {end_fmt}...")
resp = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": STATION_LAT,
        "longitude": STATION_LON,
        "start_date": start_fmt,
        "end_date": end_fmt,
        "hourly": "temperature_2m,precipitation,rain,snowfall,windspeed_10m,weathercode",
        "timezone": "America/New_York",
    },
    timeout=30,
)
resp.raise_for_status()
wj = resp.json()["hourly"]
weather = pl.DataFrame({
    "weather_time_str": wj["time"],
    "temperature_c": wj["temperature_2m"],
    "precipitation_mm": wj["precipitation"],
    "snowfall_cm": wj["snowfall"],
    "windspeed_kmh": wj["windspeed_10m"],
    "weathercode": wj["weathercode"],
}).with_columns(
    pl.col("weather_time_str").str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M").alias("weather_time")
).with_columns([
    pl.col("weather_time").dt.strftime("%Y%m%d").alias("start_date"),
    pl.col("weather_time").dt.hour().alias("hour"),
])
 
print("Reference tables ready. Starting per-day build...\n")
 
 
# =====================================================================
# 2. Per-day worker — everything created here is freed at the end of
#    the function call (plus an explicit gc.collect() as a backstop).
# =====================================================================
def build_day(d: date) -> None:
    ds = d.isoformat()
    tu_path = RT / "trip_updates" / f"{ds}.parquet"
    vp_path = RT / "vehicle_positions" / f"{ds}.parquet"
    out_path = OUT / f"{ds}.parquet"
    if out_path.exists() or not (tu_path.exists() and vp_path.exists()):
        return
 
    trip_updates = pl.read_parquet(tu_path)
    vehicle_positions = pl.read_parquet(vp_path)
 
    trip_updates = (
        trip_updates.sort("feed_timestamp")
        .group_by(["trip_id", "start_date", "stop_sequence"])
        .last()
    )
    if "schedule_relationship" in trip_updates.columns:
        trip_updates = trip_updates.filter(pl.col("schedule_relationship") == 0)
 
    trip_updates = trip_updates.with_columns(pl.from_epoch("arrival_time", time_unit="s").alias("event_time"))
    vehicle_positions = vehicle_positions.with_columns(pl.from_epoch("timestamp", time_unit="s").alias("vehicle_time"))
 
    tu_parsed = parse_trip_id(trip_updates)
    direct = tu_parsed.join(
        trips_parsed.select(["trip_id", "route_id", "direction_id", "shape_id", "service_id"]),
        on="trip_id", how="inner", suffix="_static",
    )
    unmatched = tu_parsed.join(direct.select("trip_id").unique(), on="trip_id", how="anti")
    fallback = unmatched.join(
        trips_parsed.select([
            "trip_id", "route_id", "direction_id", "shape_id", "service_id",
            "_service_day", "_origin_secs", "_trip_num",
        ]).rename({"trip_id": "_static_trip_id"}),
        on=["route_id", "_service_day", "_origin_secs", "_trip_num"],
        how="inner", suffix="_static",
    ).with_columns(pl.col("_static_trip_id").alias("trip_id")).drop("_static_trip_id")
 
    direct = direct.drop(["_service_day", "_origin_secs", "_trip_num"])
    fallback = fallback.drop(["_service_day", "_origin_secs", "_trip_num"])
    direct = (
        direct.drop(["route_id", "direction_id"])
        .rename({"route_id_static": "route_id", "direction_id_static": "direction_id"})
    )
    fallback = fallback.drop("direction_id").rename({"direction_id_static": "direction_id"})
    matched = pl.concat([direct, fallback.select(direct.columns)])
    if matched.height == 0:
        return
 
    data = matched.join(
        stop_times.select(["trip_id", "stop_sequence", "stop_id", "scheduled_arrival", "scheduled_departure"]),
        on=["trip_id", "stop_sequence"], how="inner",
    ).join(stops.select(["stop_id", "stop_lat", "stop_lon"]), on="stop_id", how="left")
 
    data = data.with_columns(pl.col("start_date").cast(pl.Int64).alias("_start_date_int"))
    data = data.join(
        service_dates.rename({"date": "_start_date_int"}),
        on=["service_id", "_start_date_int"], how="semi",
    ).drop("_start_date_int")
    if data.height == 0:
        return
 
    vp = (
        vehicle_positions.sort(["vehicle_id", "vehicle_time"])
        .with_columns([
            pl.col("latitude").shift(1).over("vehicle_id").alias("_prev_lat"),
            pl.col("longitude").shift(1).over("vehicle_id").alias("_prev_lon"),
            pl.col("vehicle_time").shift(1).over("vehicle_id").alias("_prev_time"),
        ])
    )
    vp = vp.with_columns(
        haversine_expr(pl.col("_prev_lat"), pl.col("_prev_lon"), pl.col("latitude"), pl.col("longitude")).alias("_dist_m")
    )
    vp = vp.with_columns(
        (pl.col("_dist_m") / (pl.col("vehicle_time") - pl.col("_prev_time")).dt.total_seconds().clip(lower_bound=1)).alias("speed_mps")
    )
    vp = vp.select(["vehicle_id", "vehicle_time", "latitude", "longitude", "bearing", "speed_mps"]).sort(["vehicle_id", "vehicle_time"])
 
    data = data.sort(["vehicle_id", "event_time"])
    data = data.join_asof(
        vp, left_on="event_time", right_on="vehicle_time", by="vehicle_id",
        strategy="backward", tolerance=timedelta(minutes=2),
    )
 
    data = (
        data.sort(["trip_id", "stop_sequence"])
        .with_columns(pl.col("arrival_time").shift(-1).over("trip_id").alias("next_arrival_time"))
        .with_columns((pl.col("next_arrival_time") - pl.col("arrival_time")).alias("travel_time"))
    )
    data = data.filter((pl.col("travel_time") > 0) & (pl.col("travel_time") < 1800))
    if data.height == 0:
        return
 
    data = data.with_columns(
        pl.col("event_time").dt.replace_time_zone("UTC").dt.convert_time_zone("America/New_York").alias("event_time_local")
    ).with_columns([
        pl.col("event_time_local").dt.hour().alias("hour"),
        pl.col("event_time_local").dt.weekday().alias("weekday"),
        pl.col("event_time_local").dt.month().alias("month"),
    ]).with_columns([
        ((pl.col("hour").is_between(7, 9)) | (pl.col("hour").is_between(16, 18))).cast(pl.Int8).alias("is_peak")
    ])
 
    data = data.with_columns([
        (pl.col("scheduled_arrival") - pl.col("scheduled_departure").shift(1).over("trip_id")).alias("scheduled_segment_time"),
        (pl.col("stop_sequence") / pl.col("stop_sequence").max().over("trip_id")).alias("trip_progress"),
    ])
 
    data = data.sort(["trip_id", "stop_sequence"]).with_columns(
        pl.col("stop_id").shift(-1).over("trip_id").alias("next_stop_id")
    )
    data = data.join(
        segment_network.select(["shape_id", "stop_id", "next_stop_id", "segment_length", "scheduled_travel_time"])
        .rename({"scheduled_travel_time": "segment_scheduled_travel_time"}),
        on=["shape_id", "stop_id", "next_stop_id"], how="left",
    )
    data = data.with_columns(
        (pl.col("segment_length") / pl.col("segment_scheduled_travel_time").clip(lower_bound=1)).alias("scheduled_segment_speed_mps")
    )
 
    data = data.join(
        weather.select(["start_date", "hour", "temperature_c", "precipitation_mm", "snowfall_cm", "windspeed_kmh", "weathercode"]),
        on=["start_date", "hour"], how="left",
    ).with_columns([
        (pl.col("precipitation_mm") > 0.1).cast(pl.Int8).alias("is_raining"),
        (pl.col("snowfall_cm") > 0.0).cast(pl.Int8).alias("is_snowing"),
        pl.col("weathercode").is_in([45, 48]).cast(pl.Int8).alias("is_fog"),
    ])
 
    data = data.with_columns(pl.col("start_date").str.strptime(pl.Date, "%Y%m%d").alias("service_date"))
    data = data.with_columns(
        (pl.col("service_date").cast(pl.Datetime).dt.replace_time_zone("America/New_York")
         + pl.duration(seconds=pl.col("scheduled_arrival"))).dt.epoch(time_unit="s").alias("scheduled_arrival_epoch")
    )
    data = data.with_columns((pl.col("arrival_time") - pl.col("scheduled_arrival_epoch")).alias("delay_seconds"))
 
    data = data.join(
        ridership_hourly, left_on=["route_id", "service_date", "hour"], right_on=["bus_route", "service_date", "hour"], how="left",
    ).with_columns([pl.col("ridership").fill_null(0), pl.col("transfers").fill_null(0)])
 
    data = data.sort(["trip_id", "stop_sequence"]).with_columns(
        pl.col("delay_seconds").shift(1).over("trip_id").alias("upstream_delay_seconds")
    )
 
    headway_base = (
        data.select(["route_id", "direction_id", "stop_id", "trip_id", "arrival_time"]).unique()
        .sort(["route_id", "direction_id", "stop_id", "arrival_time"])
    )
    headway_base = headway_base.with_columns(
        pl.col("arrival_time").shift(1).over(["route_id", "direction_id", "stop_id"]).alias("_prev_bus_arrival")
    ).with_columns((pl.col("arrival_time") - pl.col("_prev_bus_arrival")).alias("headway_seconds"))
    data = data.join(
        headway_base.select(["route_id", "direction_id", "stop_id", "trip_id", "arrival_time", "headway_seconds"]),
        on=["route_id", "direction_id", "stop_id", "trip_id", "arrival_time"], how="left",
    )
 
    data = data.join(calendar_features, on="service_date", how="left")
 
    model_ready = data.select(ID_COLS + FEATURES + [TARGET]).drop_nulls(subset=FEATURES + [TARGET])
    if model_ready.height > 0:
        model_ready.write_parquet(out_path)
 
    # Explicit cleanup — belt-and-suspenders on top of the function scope
    # going out of existence; matters most in a notebook kernel, less so
    # in a plain script run, but costs nothing here.
    del trip_updates, vehicle_positions, tu_parsed, direct, fallback, matched
    del data, vp, headway_base
    gc.collect()

Loading static reference tables...
Fetching weather for 2026-03-03 to 2026-07-03...
Reference tables ready. Starting per-day build...



In [4]:
n_days = (DATE_END - DATE_START).days + 1
for i in range(n_days):
    d = DATE_START + timedelta(days=i)
    build_day(d)
    if i % 10 == 0:
        print(f"  processed through {d.isoformat()}")
 
# =====================================================================
# 4. Combine day files LAZILY at the end. This reads only parquet
#    metadata + schema first; the actual collect() happens once, using
#    the streaming engine so it doesn't need all files fully materialized
#    in RAM simultaneously.
# =====================================================================
print("\nCombining daily files...")
final = pl.scan_parquet(OUT / "*.parquet").collect(streaming=True)
final.write_parquet(GTFS / "baseline_dataset.parquet")
print(f"Final dataset: {final.shape}")
print(f"On-disk size: {(GTFS / 'baseline_dataset.parquet').stat().st_size / 1e6:.1f} MB")

C:\Users\ishan\AppData\Local\Temp\ipykernel_28936\2738698008.py:169: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  data = data.join_asof(


  processed through 2026-03-03
  processed through 2026-03-13
  processed through 2026-03-23
  processed through 2026-04-02
  processed through 2026-04-12
  processed through 2026-04-22
  processed through 2026-05-02
  processed through 2026-05-12
  processed through 2026-05-22
  processed through 2026-06-01
  processed through 2026-06-11
  processed through 2026-06-21
  processed through 2026-07-01

Combining daily files...


C:\Users\ishan\AppData\Local\Temp\ipykernel_28936\2608713683.py:15: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  final = pl.scan_parquet(OUT / "*.parquet").collect(streaming=True)


Final dataset: (5596768, 40)
On-disk size: 135.9 MB


In [6]:
final.describe()

statistic,trip_id,start_date,route_id,direction_id,shape_id,service_id,stop_sequence,trip_progress,hour,weekday,month,is_peak,scheduled_arrival,scheduled_departure,scheduled_segment_time,stop_lat,stop_lon,latitude,longitude,bearing,temperature_c,precipitation_mm,snowfall_cm,windspeed_kmh,is_raining,is_snowing,is_fog,weathercode,segment_length,scheduled_segment_speed_mps,upstream_delay_seconds,speed_mps,headway_seconds,is_weekend,is_federal_holiday,is_school_day,has_major_event,ridership,transfers,travel_time
str,str,str,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""5596768""","""5596768""","""5596768""",5.596768e6,"""5596768""","""5596768""",5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6,5.596768e6
"""null_count""","""0""","""0""","""0""",0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,null,null,0.493997,null,null,30.965568,0.512843,12.783237,3.84177,4.67423,0.340268,51288.366242,51288.366242,88.800693,40.786569,-73.958888,40.786587,-73.95887,153.046524,17.111295,0.128988,0.001776,10.448009,0.117332,0.005228,0.0,10.040249,244.080253,4.311254,278.043834,14.292437,929.615452,0.229521,0.024989,0.675494,0.029468,521.634016,110.222335,91.695957
"""std""",null,null,null,0.499964,null,null,18.19807,0.275325,5.977205,1.902414,1.187777,0.473799,20862.672189,20862.672189,59.036491,0.036888,0.018155,0.036869,0.018148,95.729233,9.045848,0.490096,0.035308,5.179087,0.321815,0.072113,0.0,19.548092,131.91048,1.443005,564.841098,27.178291,1766.195145,null,null,null,null,306.4565,70.438117,77.279404
"""min""","""MV_A6-Saturday-004600_M2_201""","""20260303""","""M1""",0.0,"""M010003""","""MV_A6-Saturday""",2.0,0.025641,0.0,1.0,3.0,0.0,1200.0,1200.0,12.0,40.703036,-74.010834,40.701462,-74.01284,0.0,-6.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,61.833169,0.832685,-11534.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
"""25%""",null,null,null,0.0,null,null,16.0,0.276596,8.0,2.0,4.0,0.0,34229.0,34229.0,54.0,40.758031,-73.972348,40.758102,-73.97216,53.972626,10.4,0.0,0.0,6.7,0.0,0.0,0.0,0.0,162.396925,3.197884,-21.0,1.55598,354.0,null,null,null,null,265.0,52.0,47.0
"""50%""",null,null,null,0.0,null,null,30.0,0.513158,13.0,4.0,5.0,0.0,51612.0,51612.0,74.0,40.788413,-73.954899,40.788445,-73.954834,157.55246,17.5,0.0,0.0,10.0,0.0,0.0,0.0,3.0,216.346123,4.070214,141.0,3.943758,673.0,null,null,null,null,523.0,108.0,78.0
"""75%""",null,null,null,1.0,null,null,44.0,0.75,18.0,5.0,6.0,1.0,67151.0,67151.0,103.0,40.815025,-73.9443,40.814915,-73.944237,233.972626,24.1,0.0,0.0,13.5,0.0,0.0,0.0,3.0,271.873271,5.322992,396.0,7.940384,1079.0,null,null,null,null,756.0,162.0,112.0
"""max""","""OH_C6-Weekday-SDon-BM-141800_M…","""20260703""","""M4""",1.0,"""M150067""","""OH_C6-Weekday-SDon-BM""",77.0,0.987179,23.0,7.0,7.0,1.0,96000.0,96000.0,6198.0,40.859129,-73.926552,40.879456,-73.897079,355.236359,39.1,9.1,1.47,34.5,1.0,1.0,0.0,75.0,992.020882,15.320422,18194.0,738.841291,52695.0,1.0,1.0,1.0,1.0,1399.0,422.0,1799.0
